<div style="background-color:#000047; padding: 30px; border-radius: 10px; color: white; text-align: center;">
    <img src='Figures/alinco_white_text.png' style="height: 100px; margin-bottom: 10px;"/>
    <h1>Aprendizaje Automático Avanzado</h1>
    <h3>Ejemplo de Árboles de Decisión (Evaluación de Automóviles)</h3>
    
</div>

> En este ejemplo práctico construimos un **clasificador basado en árboles de decisión** para predecir el nivel de **seguridad/aceptabilidad de un automóvil**. Entrenaremos **dos modelos**: uno con el criterio **índice Gini** y otro con **entropía**, implementados con Python y scikit-learn. Usaremos el **Car Evaluation Data Set** del repositorio de Machine Learning de la UCI.

El algoritmo CART es la **base** de otros algoritmos importantes como los *bagged decision trees*, **Random Forest** y los *boosted decision trees* (Gradient Boosting).

En este ejemplo resolveremos un **problema de clasificación**.

**Intuición del algoritmo**

La intuición detrás del árbol de decisión es muy sencilla:

1. Para cada atributo del dataset, el algoritmo forma un **nodo**. El atributo **más importante** se coloca en el nodo raíz.
2. Para evaluar un caso, empezamos en el **nodo raíz** y descendemos por el árbol siguiendo la rama que cumple cada condición.
3. El proceso continúa hasta llegar a un **nodo hoja**, que contiene la **predicción**.

El principal reto al construir un árbol es decidir **qué atributo** usar como nodo raíz y en cada nivel. A este proceso se le llama **selección de atributos**. Las dos medidas más populares son:

### Recordando los Conceptos 
**Ganancia de información (entropía)**

La **entropía** mide la **impureza** o incertidumbre de un conjunto de ejemplos. La **ganancia de información** es la **disminución de entropía** tras dividir el dataset según un atributo. El algoritmo ID3 usa la entropía para calcular la ganancia; se elige como atributo de división el de **mayor ganancia de información**.

$$Entropía = - \sum_{i=1}^{C} p_i \log_2(p_i)$$

**Índice Gini**

El **índice Gini** es la medida que usa **CART**. Indica que, si seleccionamos dos elementos al azar de una población, deben ser de la misma clase (probabilidad 1 si la población es **pura**). Realiza únicamente **divisiones binarias**; a mayor Gini, mayor homogeneidad.

$$Gini = 1 - \sum_{i=1}^{C} p_i^2$$

Se elige como atributo de división el que produce el **menor índice Gini** (o la mayor reducción de impureza).

| Medida | Idea | Usada por |
|---|---|---|
| **Ganancia de información** | Reducción de entropía | ID3, C4.5 |
| **Índice Gini** | Probabilidad de clasificación errónea | CART |

**Planteamiento del problema**

El objetivo es **predecir la aceptabilidad/seguridad de un automóvil**. Construiremos un clasificador de árbol de decisión con Python y scikit-learn, usando el **Car Evaluation Data Set** de la UCI.

#### Descripción del dataset

El **Car Evaluation Data Set** se deriva de un modelo jerárquico de decisión desarrollado originalmente para un sistema experto. Relaciona la evaluación del automóvil (`class`) con **seis atributos de entrada**:

| Variable | Descripción | Valores |
|---|---|---|
| `buying` | Precio de compra | vhigh, high, med, low |
| `maint` | Costo de mantenimiento | vhigh, high, med, low |
| `doors` | Número de puertas | 2, 3, 4, 5more |
| `persons` | Capacidad de personas | 2, 4, more |
| `lug_boot` | Tamaño de la cajuela | small, med, big |
| `safety` | Seguridad estimada | low, med, high |
| `class` | **Objetivo:** evaluación | unacc, acc, good, vgood |

> Fuente: http://archive.ics.uci.edu/ml/datasets/Car+Evaluation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

Cargamos los datos **directamente desde el repositorio de la UCI** (así el notebook es reproducible sin archivos locales). El dataset no trae encabezados, por lo que las columnas aparecen como `0, 1, 2, ...`.

In [ ]:
# Dataset real de la UCI (se descarga al vuelo)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/car/car.data'
df = pd.read_csv(url, header=None)

# Alternativa local: df = pd.read_csv('C:/data/car.data', header=None)
df.shape

El dataset tiene **1728 instancias** y **7 variables**.

#### Análisis exploratorio de datos

#### Renombrar las columnas
El dataset no trae nombres de columna adecuados. Les asignamos nombres significativos:

#### Distribución de frecuencias de cada variable
Todas las variables son **categóricas**.

#### Explorar la variable objetivo `class`
La variable objetivo es de naturaleza **ordinal** (unacc < acc < good < vgood).

#### Valores faltantes

In [ ]:
# No hay valores faltantes en este dataset


#### Definir variables predictoras y variable objetivo

#### Dividir en conjuntos de entrenamiento y prueba

In [ ]:
from sklearn.model_selection import train_test_split



#### Ingeniería de características (codificación)

Todas las variables son **categóricas ordinales**, así que las convertimos a números con `OrdinalEncoder` de scikit-learn (evitamos dependencias externas). Los árboles de decisión **no requieren escalado**.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

encoder = OrdinalEncoder()


### Creación del modelo (Árbol de decisión con criterio **Gini**)

Entrenamos un árbol con `criterion='gini'` y profundidad máxima 3.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score



##### Comparar accuracy de entrenamiento vs. prueba
Comparamos ambos valores para detectar **sobreajuste**.

> Ambos valores son muy similares (≈ 0.79 vs ≈ 0.80), por lo que **no hay señal de sobreajuste**.

### Creación del modelo (Árbol de decisión con criterio **entropía**)

Repetimos el experimento usando `criterion='entropy'` (ganancia de información).

> Los resultados con **Gini** y **entropía** son prácticamente iguales en este dataset. Ambos criterios suelen dar árboles muy parecidos.

#### Visualización del árbol

Una gran ventaja de los árboles es que podemos **verlos**. Graficamos el árbol entrenado con criterio Gini.

In [ ]:
from sklearn.tree import plot_tree



#### Matriz de confusión

La **matriz de confusión** resume el desempeño del clasificador, mostrando aciertos y errores por cada categoría. Los cuatro resultados posibles son:

- **Verdaderos Positivos (TP):** se predice la clase y el ejemplo sí pertenece a ella.
- **Verdaderos Negativos (TN):** se predice que no pertenece y efectivamente no pertenece.
- **Falsos Positivos (FP):** se predice la clase pero el ejemplo no pertenece (**error tipo I**).
- **Falsos Negativos (FN):** se predice que no pertenece pero sí pertenece (**error tipo II**).

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


#### Reporte de clasificación

El **reporte de clasificación** muestra las métricas **precision**, **recall**, **f1-score** y **support** para cada clase.

In [ ]:
from sklearn.metrics import classification_report



1. Construimos un clasificador de árbol de decisión para predecir la aceptabilidad de un automóvil, con **dos criterios**: **Gini** y **entropía**. Ambos alcanzan un accuracy cercano a **0.80**.
2. Con el criterio **Gini**, el accuracy de entrenamiento (≈ 0.79) y el de prueba (≈ 0.80) son muy comparables → **sin sobreajuste**.
3. Con **entropía** se obtienen valores prácticamente idénticos, confirmando la estabilidad del modelo.
4. Que ambos accuracies sean tan parecidos se debe, en parte, al tamaño reducido del dataset y a la profundidad limitada (`max_depth=3`).
5. La **matriz de confusión** y el **reporte de clasificación** muestran un buen desempeño global.

>  **Recomendaciones:** probar mayor profundidad, comparar con **Random Forest** (notebook 0.5) o con **Gradient Boosting** para mejorar el desempeño.
